In [2]:
from __future__ import annotations

import argparse
import json
import os
import re
import textwrap
from pprint import pprint
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, TypedDict
import openai
import numpy as np
from pydantic import BaseModel, Field, ValidationError, model_validator
from dotenv import load_dotenv
load_dotenv()


from langchain.chat_models import ChatOpenAI


In [3]:
from typing import Optional
import requests
import json
from fastapi.exceptions import HTTPException

class Embedder:
    EMBEDDINGS_URL = os.getenv("EMBEDDINGS_URL", 'http://87.242.104.103:8080/embeddings')
    EMBEDDINGS_HEADERS = {"Content-Type": "application/json"}
    EMBEDDINGS_MODEL = os.getenv("EMBEDDING_MODEL", 'bge-m3')
    TIMEOUT = 15

    @classmethod
    def _find_embedding_in_obj(cls, obj) -> Optional[list]:
        """Recursively search for a key named 'embedding' or 'embeddings' in the JSON response."""
        if obj is None:
            return None
        if isinstance(obj, dict):
            # direct embedding
            if "embedding" in obj and isinstance(obj["embedding"], (list, tuple)):
                return obj["embedding"]
            if "embeddings" in obj and isinstance(obj["embeddings"], (list, tuple)):
                # return first embeddings entry if it's a list of vectors
                e = obj["embeddings"]
                if e and isinstance(e[0], (list, tuple)):
                    return e[0]
                return e
            # if 'data' field with list of items
            if "data" in obj and isinstance(obj["data"], (list, tuple)):
                first = obj["data"][0] if obj["data"] else None
                if first is not None:
                    # first may be dict with 'embedding'
                    if isinstance(first, dict) and "embedding" in first:
                        return first["embedding"]
                    # or a list
                    if isinstance(first, (list, tuple)):
                        return first
            # recurse
            for v in obj.values():
                found = cls._find_embedding_in_obj(v)
                if found is not None:
                    return found
        elif isinstance(obj, (list, tuple)):
            for item in obj:
                found = cls._find_embedding_in_obj(item)
                if found is not None:
                    return found
        return None

    @classmethod
    def request_to_embed_model(cls, input_query: str) -> Optional[list]:
        data = {
            "model": cls.EMBEDDINGS_MODEL,
            "input": input_query
        }

        try:
            # prefer json= so requests sets correct header and encoding
            response = requests.post(cls.EMBEDDINGS_URL, headers=cls.EMBEDDINGS_HEADERS, json=data, timeout=cls.TIMEOUT)
        except requests.exceptions.RequestException as exc:
            # network/timeout error
            raise HTTPException(status_code=502, detail=f"Embedding request failed: {exc}")

        # try to parse JSON body
        try:
            j = response.json()
        except ValueError:
            # not a json response
            raise HTTPException(status_code=502, detail=f"Embedding service returned non-JSON response: {response.text[:1000]}")

        # if service returned non-200, include returned body for diagnostics
        if response.status_code != 200:
            raise HTTPException(status_code=502, detail={
                "message": "Embedding service error",
                "status_code": response.status_code,
                "body": j,
            })

        # attempt to extract embedding from common shapes
        emb = cls._find_embedding_in_obj(j)
        if emb is None:
            # helpful diagnostics
            raise HTTPException(status_code=502, detail={
                "message": "Unable to locate embedding in response JSON",
                "response_json": j,
            })

        # normalize to plain python list[float]
        try:
            return list(map(float, emb))
        except Exception:
            # last resort: if embedding not numeric, return None
            raise HTTPException(status_code=502, detail={
                "message": "Embedding found but could not be converted to list[float]",
                "example_embedding": emb,
            })


# Test (keeps same test but wrapped to show errors)
if __name__ == "__main__":
    query = "Hello, world!"
    try:
        embedding = Embedder.request_to_embed_model(query)
        print('embedding len=', len(embedding) if embedding is not None else None)
    except Exception as e:
        print('Embedder error:', e)


embedding len= 1024


In [4]:
# Тест Embedder — вставьте в новую ячейку и выполните
try:
    emb = Embedder.request_to_embed_model("Тестовый запрос для проверки")
    print("OK: embedding length =", len(emb))
except Exception as e:
    import traceback, json
    print("Embedder failed:", e)
    traceback.print_exc()
    # если это HTTPException с .detail содержащим response_json, распечатайте его:
    try:
        detail = getattr(e, "detail", None)
        if detail:
            print("Detail:", json.dumps(detail, ensure_ascii=False, indent=2))
    except Exception:
        pass

OK: embedding length = 1024


In [5]:
# ---------- CONFIG ----------
class CONFIG:
    # Backend: "openai" | "ollama"
    LLM_BACKEND = os.getenv("LLM_BACKEND", "openai")
    # OpenAI
    CHAT_MODEL_NAME = os.getenv("CHAT_MODEL_NAME", "qwen2:72b")
    OPENAI_MODEL = os.getenv("OPENAI_MODEL", CHAT_MODEL_NAME)  # legacy name used by some code paths
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
    OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "")
    # Ollama
    OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.1:latest")
    OLLAMA_ENDPOINT = os.getenv("OLLAMA_ENDPOINT", "http://localhost:11434")

    # Embeddings
    EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "bge-m3")
    TOP_K = int(os.getenv("TOP_K", "30"))
    NEIGHBOR_EXPAND = int(os.getenv("NEIGHBOR_EXPAND", "2"))  # depth of neighbor expansion

    # Output
    DEFAULT_OUT_DIR = "./out"


In [6]:
llm = ChatOpenAI(
    api_key=CONFIG.OPENAI_API_KEY,
    base_url=CONFIG.OPENAI_BASE_URL,
    model=CONFIG.CHAT_MODEL_NAME,
    temperature=0,
    timeout=60,
)

llm.invoke("Hello!")

/tmp/ipykernel_13061/44822170.py:1: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(


AIMessage(content='Hello! How can I assist you today? Feel free to ask me any questions or let me know if you need help with anything specific.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 31, 'total_tokens': 60, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'qwen2:72b', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--0ccb5514-3546-4c5a-bb82-d0911781c10c-0')

In [7]:
# ---------- DOMAIN MODELS ----------
from typing import Any

@dataclass
class BianElement:
    id: str
    type: str
    name: str
    documentation: str = ""
    properties: Dict[str, str] = field(default_factory=dict)


@dataclass
class BianRelation:
    id: str
    type: str
    source: str
    target: str
    label: str = ""


class DataAttribute(BaseModel):
    name: str
    datatype: str = Field(..., description="string|integer|decimal|date|boolean|uuid|json")
    nullable: bool = True
    derived: bool = False
    description: Optional[str] = None


class Entity(BaseModel):
    name: str
    description: Optional[str] = None
    attributes: List[DataAttribute]
    primary_key: List[str] = Field(default_factory=list)
    source_bian_elements: List[str] = Field(default_factory=list)
    is_extension: bool = False
    extension_reason: Optional[str] = None

    @model_validator(mode='before')
    def normalize_attributes(cls, values: Any) -> Any:
        """Allow attributes to be provided as a dict mapping name->datatype
        and normalize it to the expected list[DataAttribute] representation.
        """
        if not isinstance(values, dict):
            return values
        attrs = values.get('attributes')
        if isinstance(attrs, dict):
            # convert mapping name->datatype into list of DataAttribute-like dicts
            values['attributes'] = [{'name': n, 'datatype': dt} for n, dt in attrs.items()]
        return values


class Relationship(BaseModel):
    from_entity: str
    to_entity: str
    type: str
    cardinality: str
    description: Optional[str] = None
    source_bian_elements: List[str]


class DataModel(BaseModel):
    product_name: str
    language: str = Field(default="ru")
    entities: List[Entity]
    relationships: List[Relationship]

    @model_validator(mode='after')
    def check_pk_attributes(cls, values):
        # In Pydantic v2 model_validator receives the fully constructed model as `values`.
        ents = {e.name: e for e in values.entities}
        for e in ents.values():
            if e.primary_key:
                attr_names = {a.name for a in e.attributes}
                missing = set(e.primary_key) - attr_names
                if missing:
                    raise ValueError(f"Entity {e.name} has PK not present as attributes: {missing}")
        return values


In [8]:
# ---------- KB LOADER (JSON) ----------
class JsonKB:
    def __init__(self, elements_path: str, relations_path: str):
        with open(elements_path, "r", encoding="utf-8") as f:
            raw_elements = json.load(f)
        with open(relations_path, "r", encoding="utf-8") as f:
            raw_relations = json.load(f)
        # normalize
        self.elements: Dict[str, BianElement] = {
            k: BianElement(
                id=v.get("id", k),
                type=v.get("type", "Unknown"),
                name=v.get("name", ""),
                documentation=v.get("documentation", ""),
                properties=v.get("properties", {}) or {},
            )
            for k, v in raw_elements.items()
        }
        self.relations: List[BianRelation] = [
            BianRelation(
                id=r.get("id", ""),
                type=r.get("type", "Association"),
                source=r.get("source"),
                target=r.get("target"),
                label=r.get("label", ""),
            )
            for r in raw_relations
            if r.get("source") in self.elements and r.get("target") in self.elements
        ]
        # neighbors map
        self.neighbors: Dict[str, List[str]] = {eid: [] for eid in self.elements}
        for rel in self.relations:
            self.neighbors[rel.source].append(rel.target)
            self.neighbors[rel.target].append(rel.source)


In [9]:
kb = JsonKB('out_kb/elements.json', 'out_kb/relations.json')

In [10]:
# ---------- LLM CLIENTS ----------
class LLMClient:
    def __init__(self, llm=None):
        """
        llm: optional external LLM instance (for example, langchain.chat_models.ChatOpenAI(...)).
        If provided, LLMClient will delegate requests to this instance. Otherwise falls back
        to the old openai/ollama logic.
        """
        self.backend = CONFIG.LLM_BACKEND
        self.external_llm = llm

        # If no external llm provided, ensure required packages are available for chosen backend
        if self.external_llm is None:
            if self.backend == "openai" and openai is None:
                raise RuntimeError("openai package not installed and no external llm provided")
            if self.backend == "ollama" and requests is None:
                raise RuntimeError("requests package is required for Ollama backend and no external llm provided")
            if self.backend == "openai" and CONFIG.OPENAI_API_KEY:
                openai.api_key = CONFIG.OPENAI_API_KEY

    def complete_json(self, system_prompt: str, user_prompt: str) -> str:
        # If external llm provided, delegate to it (try common interfaces)
        if self.external_llm is not None:
            try:
                return self._call_external_llm(system_prompt, user_prompt)
            except Exception as e:
                # If delegation fails, fall back to built-in backends
                print(f"[WARN] external llm call failed: {e}; falling back to backend={self.backend}")

        if self.backend == "openai":
            return self._openai_complete(system_prompt, user_prompt)
        elif self.backend == "ollama":
            return self._ollama_complete(system_prompt, user_prompt)
        else:
            raise ValueError("Unsupported backend")

    def _call_external_llm(self, system_prompt: str, user_prompt: str) -> str:
        # Try simple text-based invoke/call first
        combined = system_prompt + "\n" + user_prompt
        # Common LangChain ChatOpenAI might implement `invoke`, `__call__` or `generate`.
        if hasattr(self.external_llm, "invoke"):
            return self.external_llm.invoke(combined)
        if callable(self.external_llm):
            # some wrappers are callable
            try:
                return self.external_llm(combined)
            except Exception:
                pass
        # Try LangChain-style generate with HumanMessage/SystemMessage
        try:
            from langchain.schema import HumanMessage, SystemMessage
            resp = self.external_llm.generate([[SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]])
            # extract text from response (LangChain Generate -> generations structure)
            if hasattr(resp, "generations") and resp.generations:
                return resp.generations[0][0].text
        except Exception:
            pass
        raise RuntimeError("External llm does not support known call interfaces")

    def _openai_complete(self, system_prompt: str, user_prompt: str) -> str:
        resp = openai.chat.completions.create(
            model=CONFIG.CHAT_MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.1,
        )
        return resp.choices[0].message.content

    def _ollama_complete(self, system_prompt: str, user_prompt: str) -> str:
        url = f"{CONFIG.OLLAMA_ENDPOINT}/api/chat"
        payload = {
            "model": CONFIG.OLLAMA_MODEL,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            "options": {"temperature": 0.1}
        }
        r = requests.post(url, json=payload, timeout=300)
        r.raise_for_status()
        data = r.json()
        if isinstance(data, dict) and "message" in data:
            return data["message"]["content"]
        if isinstance(data, list):
            texts = [chunk.get("message", {}).get("content", "") for chunk in data]
            return "".join(texts)
        return json.dumps(data)


In [11]:
# ---------- PROMPTS ----------
SYSTEM_PROMPT = (
    "You are a senior banking data modeler. You MUST align your output to the provided "
    "BIAN context. Do not invent entities or attributes if they are not present or cannot "
    "be justified by BIAN elements. If something is needed but missing, add it with "
    "is_extension=true and a brief extension_reason. Output STRICT JSON matching the schema: "
    "DataModel(product_name, language, entities[], relationships[]). Use Russian field names when appropriate."
)

USER_PROMPT_TEMPLATE = """
Пользовательский запрос (RU):
{request}

Контекст BIAN (релевантные элементы):
{context}

Требования к результату:
1) Верни ТОЛЬКО корректный JSON по схеме: DataModel.
2) Для каждой сущности укажи source_bian_elements (список ID из контекста), primary_key (минимум 1 поле), attributes (тип: string|integer|decimal|date|boolean|uuid|json).
3) Для каждой связи заполни: from_entity, to_entity, type (association|composition|aggregation|identifying|reference), cardinality (1..1|1..*|0..1|0..*), source_bian_elements.
4) Не используй сущности/поля без обоснования в BIAN — пометь их как is_extension=true с кратким extension_reason.
5) product_name = краткое русское имя продукта.
"""


In [12]:
# ---------- UTIL ----------
JSON_FENCE = re.compile(r"\{[\s\S]*\}\s*$")

def extract_json_block(text: str) -> str:
    t = text.strip()
    if t.startswith("{") and t.endswith("}"):
        return t
    m = JSON_FENCE.search(t)
    if m:
        return m.group(0)
    return t.replace("```json", "").replace("```", "").strip()

In [13]:
# ---------- PIPELINE STATE (LangGraph) ----------
class AgentState(TypedDict):
    request: str
    retrieved_context: str
    draft_model: dict
    validated_model: dict
    artifacts: Dict[str, str]

In [14]:
# ---------- RETRIEVER ----------
# Provide a fallback SimpleEmbeddingIndex implementation if not present in the environment.
try:
    SimpleEmbeddingIndex  # type: ignore
except NameError:
    class SimpleEmbeddingIndex:
        """Lightweight in-memory embedding index using Embedder.request_to_embed_model and numpy.
        Used as a fallback when a more advanced index is not available.
        """
        def __init__(self, model_name: str | None = None):
            self.model_name = model_name
            self.ids: list[str] = []
            self.texts: list[str] = []
            self.vectors: list = []

        def add(self, ids: list[str], texts: list[str]):
            for eid, txt in zip(ids, texts):
                emb = Embedder.request_to_embed_model(txt)
                if emb is None:
                    continue
                self.ids.append(eid)
                self.texts.append(txt)
                self.vectors.append(np.array(emb, dtype=float))

        def build(self):
            # no-op for this simple index
            return

        def search(self, query: str, top_k: int = 10) -> list[tuple[str, float]]:
            if not self.vectors:
                return []
            qv = Embedder.request_to_embed_model(query)
            if qv is None:
                return []
            qv = np.array(qv, dtype=float)
            vecs = np.vstack(self.vectors)
            denom = (np.linalg.norm(vecs, axis=1) * (np.linalg.norm(qv) + 1e-12)) + 1e-12
            sims = (vecs @ qv) / denom
            idx = np.argsort(-sims)[:top_k]
            return [(self.ids[i], float(sims[i])) for i in idx]

class Retriever:
    def __init__(self, kb: JsonKB):
        self.kb = kb
        self.index = SimpleEmbeddingIndex(CONFIG.EMBEDDING_MODEL)
        ids, texts = [], []
        for eid, el in kb.elements.items():
            # attach neighbor names for context
            neigh_names = ", ".join([kb.elements[nid].name for nid in kb.neighbors.get(eid, [])[:20] if nid in kb.elements])
            buff = textwrap.dedent(f"""
                ID: {eid}
                Type: {el.type}
                Name: {el.name}
                Documentation: {el.documentation}
                Neighbors: {neigh_names}
            """)
            ids.append(eid)
            texts.append(buff)
        self.index.add(ids, texts)
        self.index.build()

    def query(self, request: str, top_k: int) -> List[str]:
        # Simple RU→EN synonyms/expansion for loans
        expansions = {
            "автокредит": "auto loan vehicle loan car finance",
            "кредит": "loan lending",
            "залог": "collateral pledge security",
            "процентная ставка": "interest rate pricing",
            "график платежей": "repayment schedule amortization",
        }
        extra = []
        for k, v in expansions.items():
            if k in request.lower():
                extra.append(v)
        q = request + (" " + " ".join(extra) if extra else "")
        hits = self.index.search(q, top_k=top_k)
        ctx = []
        for eid, score in hits:
            el = self.kb.elements[eid]
            ctx.append(f"[{eid}] {el.type} :: {el.name}\n{el.documentation}\n")
        # Expand with 1-hop neighbors of top N
        expand_ids = {eid for eid, _ in hits[:10]} if isinstance(hits, list) else set()
        for eid in list(expand_ids):
            for n in self.kb.neighbors.get(eid, [])[:20]:
                expand_ids.add(n)
        # append neighbor summaries
        for n in list(expand_ids):
            if n not in {eid for eid, _ in hits}:
                el = self.kb.elements.get(n)
                if el:
                    ctx.append(f"[{n}] {el.type} :: {el.name}\n{el.documentation}\n")
        return ctx


In [15]:
# ---------- MILVUS RETRIEVER (pymilvus) ----------
try:
    from pymilvus import (
        connections,
        FieldSchema,
        CollectionSchema,
        DataType,
        Collection,
        utility,
    )
except Exception:
    connections = None

class MilvusRetriever:
    """Retriever backed by a local Milvus instance using pymilvus.

    Features:
    - Builds a collection of embeddings for the provided JsonKB using the
      existing Embedder.request_to_embed_model.
    - Inserts vectors in batches and creates an index.
    - Keeps an in-memory map of integer pk -> element id.
    - Provides `query(request, top_k)` -> List[str] returning context strings
      (same format as the existing Retriever.query).

    Notes:
    - This implementation keeps explicit INT64 primary keys (auto_id=False)
      and manages them locally to avoid differences between pymilvus versions
      in how auto-generated ids are returned.
    - If you prefer Milvus to manage primary keys, change the schema to
      auto_id=True and adapt the insert -> returned primary keys mapping.
    """

    def __init__(self, kb: JsonKB, collection_name: str = "kb_embeddings", host: str = "127.0.0.1", port: str = "19530", dim: Optional[int] = None, rebuild: bool = False):
        if connections is None:
            raise RuntimeError("pymilvus is required for MilvusRetriever. Install pymilvus in your venv.")
        self.kb = kb
        self.collection_name = collection_name
        self.host = host
        self.port = port
        self.dim = dim
        self.id_map: dict[int, str] = {}
        self._collection: Collection | None = None

        # connect
        connections.connect(host=self.host, port=self.port)

        # optionally drop existing collection
        if utility.has_collection(self.collection_name) and rebuild:
            try:
                Collection(self.collection_name).drop()
            except Exception:
                pass

        # create or open collection
        if not utility.has_collection(self.collection_name):
            # if dim known we can create schema now, otherwise defer to build_index
            if self.dim is None:
                self._collection = None
            else:
                fields = [
                    FieldSchema(name="pk", dtype=DataType.INT64, is_primary=True, auto_id=False),
                    FieldSchema(name="eid", dtype=DataType.VARCHAR, max_length=512),
                    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=self.dim),
                ]
                schema = CollectionSchema(fields, description="KB element embeddings")
                self._collection = Collection(self.collection_name, schema)
        else:
            self._collection = Collection(self.collection_name)
            # try to infer dim from schema
            try:
                emb_field = [f for f in self._collection.schema.fields if f.dtype == DataType.FLOAT_VECTOR][0]
                self.dim = getattr(emb_field, "dim", None) or emb_field.params.get("dim")
            except Exception:
                pass

        # load collection if available
        if self._collection is not None:
            try:
                self._collection.load()
            except Exception:
                pass

    def build_index(self, batch_size: int = 64, index_params: dict | None = None):
        """Compute embeddings for all KB elements and insert into Milvus.

        - batch_size: number of vectors to insert per request
        - index_params: index description passed to `create_index`
        """
        if index_params is None:
            index_params = {"index_type": "IVF_FLAT", "metric_type": "L2", "params": {"nlist": 128}}

        # prepare texts/eids
        eids: list[str] = []
        texts: list[str] = []
        for eid, el in self.kb.elements.items():
            neigh_names = ", ".join([self.kb.elements[nid].name for nid in self.kb.neighbors.get(eid, [])[:20] if nid in self.kb.elements])
            buff = f"{el.name} {el.documentation} {neigh_names}".strip()
            eids.append(eid)
            texts.append(buff)

        if not texts:
            return

        # infer dim and create collection if needed
        if self.dim is None:
            sample = Embedder.request_to_embed_model(texts[0])
            if sample is None:
                raise RuntimeError("Failed to obtain a sample embedding; check Embedder configuration.")
            self.dim = len(sample)
            fields = [
                FieldSchema(name="pk", dtype=DataType.INT64, is_primary=True, auto_id=False),
                FieldSchema(name="eid", dtype=DataType.VARCHAR, max_length=512),
                FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=self.dim),
            ]
            schema = CollectionSchema(fields, description="KB element embeddings")
            self._collection = Collection(self.collection_name, schema)

        # determine starting primary key to avoid collisions
        next_pk = 1
        try:
            if self._collection is not None:
                existing = int(self._collection.num_entities)
                next_pk = existing + 1
        except Exception:
            pass

        # insert in batches, maintain id_map
        pks = []
        e_ids = []
        vectors = []
        for i, txt in enumerate(texts):
            emb = Embedder.request_to_embed_model(txt)
            if emb is None:
                continue
            pks.append(next_pk)
            e_ids.append(eids[i])
            vectors.append(emb)
            self.id_map[next_pk] = eids[i]
            next_pk += 1

            # flush
            if len(pks) >= batch_size or i == len(texts) - 1:
                # insert takes a list of columns in the same order as schema fields
                try:
                    self._collection.insert([pks, e_ids, vectors])
                except Exception as ex:
                    # try a more permissive insertion format if needed
                    self._collection.insert({"pk": pks, "eid": e_ids, "embedding": vectors})
                pks = []
                e_ids = []
                vectors = []

        # Ensure all data is persisted before creating an index
        self._collection.flush()

        # create index on embedding and load collection
        try:
            self._collection.create_index("embedding", index_params)
        except Exception:
            # index may already exist or backend refused params
            pass
        try:
            self._collection.load()
        except Exception:
            pass

    def query(self, request: str, top_k: int) -> list[str]:
        """Return context list similar to Retriever.query"""
        if self._collection is None:
            raise RuntimeError("Collection not initialized. Call build_index() first.")
        vec = Embedder.request_to_embed_model(request)
        if vec is None:
            return []

        # milvus search params; try common `param` or `params` keywords depending on pymilvus version
        search_params = {"metric_type": "L2", "params": {"nprobe": 10}}
        try:
            res = self._collection.search([vec], "embedding", param=search_params, limit=top_k)
        except TypeError:
            # fallback to `params` keyword
            res = self._collection.search([vec], "embedding", params=search_params, limit=top_k)

        ctx: list[str] = []
        hit_eids: set[str] = set()

        # res is a list of SearchResult for each query
        for hit in res[0]:
            try:
                pk_val = int(hit.id)
            except Exception:
                pk_val = int(getattr(hit, "id", hit))
            eid = self.id_map.get(pk_val)
            if eid is None:
                continue
            el = self.kb.elements.get(eid)
            if not el:
                continue
            ctx.append(f"[{eid}] {el.type} :: {el.name}\n{el.documentation}\n")
            hit_eids.add(eid)

        # Expand with 1-hop neighbors of top results
        expand_ids = set(hit_eids)
        for eid in list(expand_ids):
            for n in self.kb.neighbors.get(eid, [])[:20]:
                expand_ids.add(n)
        for n in list(expand_ids):
            if n not in hit_eids:
                el = self.kb.elements.get(n)
                if el:
                    ctx.append(f"[{n}] {el.type} :: {el.name}\n{el.documentation}\n")
        return ctx

    def drop(self):
        """Drop the collection from Milvus if it exists."""
        try:
            if utility.has_collection(self.collection_name):
                Collection(self.collection_name).drop()
        except Exception:
            pass

    def num_entities(self) -> int:
        try:
            if self._collection is None:
                return 0
            return int(self._collection.num_entities)
        except Exception:
            return 0


In [16]:
# ---------- RENDERERS ----------
SQL_TYPE_MAP = {
    'string': 'text',
    'integer': 'integer',
    'decimal': 'numeric',
    'date': 'date',
    'boolean': 'boolean',
    'uuid': 'uuid',
    'json': 'jsonb'
}

def render_plantuml(model: DataModel) -> str:
    lines = ["@startuml", "hide circle"]
    # Assign stable ASCII aliases for entities: E1, E2, ...
    alias_map = {}
    for idx, e in enumerate(model.entities, start=1):
        alias = f"E{idx}"
        alias_map[e.name] = alias
        lines.append(f"entity \"{e.name}\" as {alias} {{")
        for a in e.attributes:
            null = '' if not a.nullable else ' (nullable)'
            lines.append(f"  {a.name} : {a.datatype}{null}")
        if e.primary_key:
            lines.append("  -- PK --")
            for pk in e.primary_key:
                lines.append(f"  * {pk}")
        lines.append("}")
    # Render relationships using aliases; fallback to sanitized id if target not found
    for r in model.relationships:
        arrow = {
            'association': '--',
            'reference': '..',
            'composition': '*--',
            'aggregation': 'o--',
            'identifying': '+--'
        }.get(r.type, '--')
        from_alias = alias_map.get(r.from_entity, sanitize_id(r.from_entity))
        to_alias = alias_map.get(r.to_entity, sanitize_id(r.to_entity))
        lines.append(f"{from_alias} {arrow} {to_alias} : {r.cardinality}")
    lines.append("@enduml")
    return "\n".join(lines)

def render_sql(model: DataModel) -> str:
    ddl = []
    for e in model.entities:
        cols = []
        pk = ''
        for a in e.attributes:
            sql_type = SQL_TYPE_MAP.get(a.datatype.lower(), 'text')
            null = ' NOT NULL' if not a.nullable else ''
            cols.append(f"\t{sql_ident(a.name)} {sql_type}{null}")
        if e.primary_key:
            pk = f",\n\tPRIMARY KEY ({', '.join(sql_ident(x) for x in e.primary_key)})"
        ddl.append(f"CREATE TABLE {sql_ident(e.name)} (\n" + ",\n".join(cols) + pk + "\n);\n")
    for r in model.relationships:
        if r.cardinality.strip() in {"1..*", "0..*"}:
            parent = next((e for e in model.entities if e.name == r.from_entity), None)
            child = next((e for e in model.entities if e.name == r.to_entity), None)
            if parent and child and parent.primary_key:
                fk_cols = ', '.join(sql_ident(x) for x in parent.primary_key)
                ddl.append(
                    f"ALTER TABLE {sql_ident(child.name)}\n"
                    f"  ADD CONSTRAINT fk_{sanitize_id(child.name)}_{sanitize_id(parent.name)}\n"
                    f"  FOREIGN KEY ({fk_cols}) REFERENCES {sql_ident(parent.name)}({fk_cols});\n"
                )
    return "\n".join(ddl)

def sanitize_id(name: str) -> str:
    return re.sub(r"[^A-Za-zА-Яа-я0-9_]", "_", name)

def sql_ident(name: str) -> str:
    safe = re.sub(r"[^A-Za-zА-Яа-я0-9_]", "_", name)
    return '"' + safe + '"'

In [17]:
# ---------- NODES ----------
def node_retrieve(state: AgentState, retriever: Retriever) -> AgentState:
    ctx_list = retriever.query(state["request"], CONFIG.TOP_K)
    state["retrieved_context"] = "\n".join(ctx_list)
    # log retrieved context
    print('===CONTEXT===')
    pprint("\n".join(ctx_list))
    return state

class LLMClientWrapper:
    def __init__(self):
        self.client = LLMClient()

    def generate(self, request: str, context: str) -> dict:
        user_prompt = USER_PROMPT_TEMPLATE.format(request=request, context=context)
        raw = self.client.complete_json(SYSTEM_PROMPT, user_prompt)
        text = extract_json_block(raw)
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            fix_prompt = "Исправь JSON так, чтобы он строго парсился в Python json.loads без комментариев и лишнего текста.\n" + text
            fixed = self.client.complete_json("Ты валидатор JSON.", fix_prompt)
            return json.loads(extract_json_block(fixed))


def node_generate(state: AgentState, llm: LLMClientWrapper) -> AgentState:
    data = llm.generate(state["request"], state["retrieved_context"]) 
    state["draft_model"] = data
    return state


def node_validate(state: AgentState) -> AgentState:
    model = DataModel(**state["draft_model"]) 
    state["validated_model"] = json.loads(model.json())
    return state


def node_render(state: AgentState) -> AgentState:
    model = DataModel(**state["validated_model"]) 
    # pydantic v2's model_dump_json does not accept ensure_ascii; produce UTF-8 JSON via json.dumps
    state["artifacts"] = {
        "json": json.dumps(model.model_dump(), ensure_ascii=False, indent=2),
        "puml": render_plantuml(model),
        "sql": render_sql(model),
    }
    return state


In [18]:
def node_expand(state: AgentState, llm: LLMClientWrapper) -> AgentState:
    """Generate English related search terms for the user's request using the LLM.

    Stores result in state['expansions'] as list[str]. Falls back to a static map if LLM fails.
    """
    system = (
        "You are an assistant that, given a short Russian banking product query, returns a JSON array of up to 10 short English search phrases or keywords relevant for vector search. "
        "OUTPUT ONLY the JSON array (e.g. [\"auto loan\", \"vehicle loan\"])."
    )
    user = f"Запрос: {state['request']}\nReturn a JSON array of English search terms (strings)."
    terms = []
    try:
        raw = llm.client.complete_json(system, user)
        text = extract_json_block(raw)
        arr = json.loads(text)
        if isinstance(arr, list):
            # sanitize
            terms = [str(x).strip() for x in arr if x is not None]
        else:
            raise ValueError("LLM returned non-list")
    except Exception as e:
        # fallback static expansions map
        fallback = {
            "автокредит": "auto loan vehicle loan car finance",
            "кредит": "loan lending",
            "залог": "collateral pledge security",
            "процентная ставка": "interest rate pricing",
            "график платежей": "repayment schedule amortization",
        }
        key = state.get("request", "").lower()
        suggested = fallback.get(key, "")
        if suggested:
            terms = [t for t in suggested.split() if t]
        else:
            # minimal fallback: use request itself (will be searched as-is)
            terms = [state.get("request", "")] if state.get("request") else []
    # deduplicate while preserving order
    seen = set()
    out = []
    for t in terms:
        if t and t not in seen:
            seen.add(t)
            out.append(t)
    state["expansions"] = out
    print("EXPANSIONS:", out)
    return state


In [42]:
request = "ипотека"
print(f"Установлен запрос: '{request}'")

Установлен запрос: 'ипотека'


In [43]:
from langgraph.graph import StateGraph, END

# Build graph if available, else linear
state: AgentState = {
    "request": request,
    "retrieved_context": "",
    "draft_model": {},
    "validated_model": {},
    "artifacts": {},
}

In [23]:
# ---------- SETUP: KB, RETRIEVER, LLM ----------
# Prefer MilvusRetriever when available; fall back to in-memory Retriever.
try:
    retriever = None
    try:
        mr = MilvusRetriever(kb, collection_name='kb_embeddings', rebuild=False)
        # If collection exists but is empty, build index now (safe default).
        if mr.num_entities() == 0:
            print('Milvus collection empty — building index (this may take a while)...')
            mr.build_index(batch_size=64)
        retriever = mr
        print(f'Using MilvusRetriever(collection={mr.collection_name}, entities={mr.num_entities()})')
    except Exception as e:
        # Milvus not available or failed to initialize — will fall back
        print('MilvusRetriever not available:', e)

    if retriever is None:
        retriever = Retriever(kb)
        print('Falling back to in-memory Retriever')
except Exception as exc:
    # Keep notebook usable even if unexpected errors occur
    print('Error while setting up retriever:', exc)
    retriever = Retriever(kb)

# LLM wrapper (unchanged)
llm = LLMClientWrapper()


Using MilvusRetriever(collection=kb_embeddings, entities=178)


In [24]:
kb.elements.__len__(), kb.relations.__len__()

(178, 389)

In [25]:
mr.num_entities()

178

In [26]:
g = StateGraph(AgentState)
g.add_node("retrieve", lambda s: node_retrieve(s, retriever))
g.add_node("generate", lambda s: node_generate(s, llm))
g.add_node("validate", node_validate)
g.add_node("render", node_render)
g.set_entry_point("retrieve")
g.add_edge("retrieve", "generate")
g.add_edge("generate", "validate")
g.add_edge("validate", "render")
g.add_edge("render", END)
app = g.compile()

In [47]:
# Rebuild graph to include 'expand' node
g = StateGraph(AgentState)
g.add_node("expand", lambda s: node_expand(s, llm))
g.add_node("retrieve", lambda s: node_retrieve(s, retriever))
g.add_node("generate", lambda s: node_generate(s, llm))
g.add_node("validate", node_validate)
g.add_node("render", node_render)
# entry point is expand
g.set_entry_point("expand")
# edges: expand -> retrieve -> generate -> validate -> render
g.add_edge("expand", "retrieve")
g.add_edge("retrieve", "generate")
g.add_edge("generate", "validate")
g.add_edge("validate", "render")
g.add_edge("render", END)
app = g.compile()
print('Graph compiled with entry point: expand')


Graph compiled with entry point: expand


In [48]:
state = app.invoke(state)

EXPANSIONS: ['mortgage', 'home loan', 'real estate financing', 'property loan', 'housing finance']
===CONTEXT===
''


/tmp/ipykernel_13061/164477807.py:34: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  state["validated_model"] = json.loads(model.json())


In [45]:
# --- TEST: expand node + retriever integration
# Run expand node standalone
state = {
    "request": request,
    "retrieved_context": "",
    "draft_model": {},
    "validated_model": {},
    "artifacts": {},
}
state = node_expand(state, llm)
print('\nGenerated expansions:', state.get('expansions'))

# Query retriever for each expansion term and show top-3 hits
for term in state.get('expansions', [])[:10]:
    print(f"\n=== Query term: '{term}' ===")
    try:
        hits = retriever.query(term, top_k=3)
        for h in hits:
            print('-', h)
    except Exception as e:
        print('Retriever error for term', term, e)


EXPANSIONS: ['mortgage', 'home loan', 'real estate financing', 'property loan', 'housing finance']

Generated expansions: ['mortgage', 'home loan', 'real estate financing', 'property loan', 'housing finance']

=== Query term: 'mortgage' ===

=== Query term: 'home loan' ===

=== Query term: 'real estate financing' ===

=== Query term: 'property loan' ===

=== Query term: 'housing finance' ===


In [49]:
state

{'request': 'ипотека',
 'retrieved_context': '',
 'draft_model': {'product_name': 'Ипотека',
  'language': 'ru',
  'entities': [{'name': 'Заемщик',
    'source_bian_elements': ['Customer'],
    'primary_key': ['id'],
    'attributes': [{'name': 'id', 'datatype': 'uuid'},
     {'name': 'имя', 'datatype': 'string'},
     {'name': 'фамилия', 'datatype': 'string'},
     {'name': 'отчество', 'datatype': 'string'},
     {'name': 'дата_рождения', 'datatype': 'date'},
     {'name': 'адрес', 'datatype': 'string'},
     {'name': 'телефон', 'datatype': 'string'},
     {'name': 'email', 'datatype': 'string'}]},
   {'name': 'Ипотечный_кредит',
    'source_bian_elements': ['Loan'],
    'primary_key': ['id'],
    'attributes': [{'name': 'id', 'datatype': 'uuid'},
     {'name': 'номер_кредита', 'datatype': 'string'},
     {'name': 'сумма_кредита', 'datatype': 'decimal'},
     {'name': 'процентная_ставка', 'datatype': 'decimal'},
     {'name': 'срок_кредита', 'datatype': 'integer'},
     {'name': 'дата

In [50]:
# export to plant uml file
with open("out_model.puml", "w", encoding="utf-8") as f:
    f.write(state['artifacts']['puml'])

In [51]:
# export to json file
with open("out_model.json", "w", encoding="utf-8") as f:
    json.dump(state['artifacts']['json'], f, ensure_ascii=False, indent=2)

# export to ddl 
with open("out_model.sql", "w", encoding="utf-8") as f:
    f.write(state['artifacts']['sql'])